<a href="https://colab.research.google.com/github/Nitsh-kumar/LangChain_learning/blob/main/Langchain_learning_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.3 MB/s eta 0:00:00


In [ ]:

import os
from google.colab import userdata
from langchain_core.prompts import PromptTemplate,ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import JsonOutputParser
from langchain_groq import ChatGroq
from pprint import pprint
from langsmith import traceable

In [ ]:
# Setting up the environment variables and API's
#LLM
os.environ['GROQ_API_KEY']  = userdata.get('GROQ_API_KEY')

#Langsmith for learning and tracking
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = userdata.get('langsmith_api')
os.environ["LANGSMITH_PROJECT"] = "Learning_langsmith"
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'


In [ ]:
llm = ChatGroq(
    model = 'openai/gpt-oss-20b',
    # temperature= 0.3,
    streaming= True,
)

There are Many ways to get stuctured output such as:


*   Json output
*   Pydantic Model output(validated output most widely used)
* list output


In [ ]:
# first we will cover Json output

# Scenerio:    A company receives thousands of support tickets daily.
# They need to:
# Classify the issue
# Assign priority
# Decide routing
# …and log everything for audit & debugging.

In [ ]:
import json

#  we need to define a particular json schema to validate the structure

ticket_schema = {
    "title": "TicketClassification",
    "description": "Structured classification of a customer support ticket",
    "type": "object",
    "properties": {
        "category": {
            "type": "string",
            "enum": [
                "billing",
                "technical",
                "account",
                "feature_request",
                "other"
            ],
            "description": "Primary issue category"
        },
        "priority": {
            "type": "string",
            "enum": ["low", "medium", "high"],
            "description": "Urgency of the ticket"
        },
        "requires_human": {
            "type": "boolean",
            "description": "Whether a human agent must handle this"
        },
        "short_reason": {
            "type": "string",
            "description": "One-line justification for the decision"
        }
    },
    "required": [
        "category",
        "priority",
        "requires_human",
        "short_reason"
    ],
    "additionalProperties": False
}

In [ ]:
@traceable(
    name="Ticket Classification (JSON Schema)",
    run_type="chain",
    tags=["support", "classification", "json-schema"],
    metadata={"schema_version": "v1"}
)
def classify_ticket_json(ticket_text: str) -> dict:
    prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            "You are a customer support classifier. "
            "Return ONLY valid JSON that strictly follows the provided schema."
        ),
        (
            "human",
            "Support ticket:\n\n{ticket_text}"
        )
    ])

    pipeline = prompt | llm.with_structured_output(
        schema=ticket_schema,
        method="json_schema"   # 🔑 important
    )

    return pipeline.invoke({"ticket_text": ticket_text})

In [ ]:
ticket = """
My card was charged twice and I can't see the invoice.
Please resolve this urgently.
"""

result = classify_ticket_json(ticket)
pprint(result)

{'category': 'billing',
 'priority': 'high',
 'requires_human': True,
 'short_reason': 'Duplicate charge and missing invoice'}


# 🧩 JSON Structured Output in LangChain — Notes

---

## 1️⃣ What JSON Structured Output Is

**JSON Structured Output** means forcing an LLM to return data that:

- Is valid JSON
- Follows a predefined **JSON Schema**
- Has a predictable shape
- Can be safely consumed by code

👉 The output is **machine-first**, not human-first.

---

## 2️⃣ Why JSON Structured Output Matters in Industry

Free-text LLM outputs are:
- Hard to parse
- Error-prone
- Unsafe for automation

JSON structured output enables:

- ✅ Deterministic pipelines  
- ✅ Strong validation  
- ✅ Reliable automation  
- ✅ Safe downstream processing  
- ✅ Auditable AI decisions  

This is **mandatory** in production systems.

---

## 3️⃣ JSON Schema vs Free JSON

Not all JSON is equal.

### Free JSON
- Model *tries* to output JSON
- No guarantees
- Fields may be missing or extra
- Types may be wrong

### JSON Schema–based Output
- Output must match a **formal schema**
- Required fields are enforced
- Extra fields can be blocked
- Types are validated

👉 JSON Schema turns LLM output into a **contract**.

---

## 4️⃣ What JSON Schema Represents

JSON Schema defines:

- Allowed fields
- Field data types
- Required vs optional fields
- Allowed values (`enum`)
- Whether extra fields are allowed

It answers:

> “What shape of data is valid — regardless of language?”

---

## 5️⃣ Why JSON Schema Is Used Over Pydantic (Sometimes)

### JSON Schema is preferred when:

- Multiple languages are involved
- Output crosses service boundaries
- APIs are exposed publicly
- Frontend + backend share contracts
- Compliance / audit teams are involved

### Pydantic is preferred when:
- System is Python-only
- Output is consumed immediately
- Developer velocity matters

👉 **JSON Schema = system contract**  
👉 **Pydantic = Python convenience**

---

## 6️⃣ How LangChain Supports JSON Structured Output

LangChain can:
- Accept a JSON Schema directly
- Instruct the LLM to follow it
- Validate the response
- Return a Python `dict`

If validation fails:
- An error is raised
- The failure is traceable (via LangSmith)

This makes failures **visible**, not silent.

---

## 7️⃣ Strict vs Best-Effort JSON Schema (Important)

### Strict Mode
- Output **must** match schema
- Generation-time enforcement
- Best for production

### Best-Effort Mode
- Model attempts to follow schema
- Validation still applied after
- Useful for experimentation

👉 Always prefer **strict mode** when available.

---

## 8️⃣ JSON Structured Output + LangSmith (Why This Combo Matters)

When combined with LangSmith, you get:

- Full trace of inputs
- Prompt messages
- JSON schema used
- Final validated JSON output
- Errors on schema mismatch
- Schema version tracking

This enables:
- Debugging
- Auditing
- Model evaluation
- Compliance reporting

---

## 9️⃣ Real Industry Use Cases

JSON structured output is used in:

- Support ticket classification
- Fraud detection decisions
- Content moderation
- Tool routing / agent decisions
- RAG answer formatting
- API response generation
- Workflow orchestration

Anywhere **code depends on LLM output**, JSON structure is required.

---

## 🔟 Common Mistakes to Avoid

- ❌ Asking “return JSON” without schema
- ❌ Parsing LLM output with regex
- ❌ Allowing extra fields unknowingly
- ❌ Skipping validation
- ❌ Treating free text as reliable data

---

## 1️⃣1️⃣ Schema Design Best Practices

- Keep schemas small and focused
- Use `enum` wherever possible
- Mark required fields explicitly
- Set `additionalProperties: false`
- Version schemas (`v1`, `v2`)
- Store schemas in source control

---

## 🧠 Final Mental Model

> **Free text is for humans.  
> JSON Schema is for systems.  
> LangChain enforces it.  
> LangSmith explains it.**

---

## TL;DR

- JSON structured output = reliable LLM automation
- JSON Schema defines the contract
- LangChain enforces the schema
- LangSmith provides traceability
- This pattern is production-grade AI

---


In [ ]:
# Now we will focus on the star of the show, pydantic model validator

# from pydantic import BaseModel , EmailStr
# from typing import Literal ,  Field

from pydantic import BaseModel, Field
from typing import List, Literal

class StudentScreeningResult(BaseModel):
    candidate_fit: Literal[
        "strong_fit",
        "moderate_fit",
        "weak_fit"
    ]

    score: int = Field(
        ge=0,
        le=100,
        description="Overall suitability score between 0 and 100"
    )

    matched_skills: List[str] = Field(
        description="Skills present in both the job description and the resume"
    )

    missing_skills: List[str] = Field(
        description="Important job skills missing from the resume"
    )

    experience_level: Literal[
        "junior",
        "mid",
        "senior"
    ]

    shortlist: bool = Field(
        description="Whether the candidate should be shortlisted"
    )

    decision_reason: str = Field(
        description="Brief explanation for the screening decision"
    )




In [ ]:
resume_screening_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are an automated resume screening system.
              Your task is to evaluate a candidate resume against a job description
              and return a structured decision.

              Rules:
              - Be objective and conservative.
              - Base decisions ONLY on the provided inputs.
              - Do not assume missing information.
              - Do not invent skills or experience.
              - Follow the output schema strictly.
              - If information is unclear, reflect that in the score and reasoning.""".strip()
        ),
        (
            "human",
            """Job Description:
            {job_description}
            Candidate Resume:
            {resume_text}
            Evaluate the candidate based on:
            - Skill match
            - Experience relevance
            - Seniority level

            Return a structured assessment.""".strip()
        ),
    ]
)


In [ ]:
@traceable(
    name="Resume Screening Decision",
    run_type="chain",
    tags=["resume-screening", "pydantic-structured-output"],
    metadata={"schema_version": "v1"}
)
def screen_candidate(job_description: str,resume_text: str) -> StudentScreeningResult:
    pipeline = (
        resume_screening_prompt
        | llm.with_structured_output(StudentScreeningResult)
    )

    return pipeline.invoke(
        {
            "job_description": job_description,
            "resume_text": resume_text,
        }
    )


In [ ]:
job_description = """
Looking for a Machine Learning Engineer with strong Python skills,
experience in ML models, AWS, and production deployment.
"""

resume_text = """
Software Engineer with 3 years of experience in Python.
Worked on ML models for classification tasks.
Some exposure to cloud platforms.
"""

result = screen_candidate(job_description, resume_text)

pprint(result)

StudentScreeningResult(candidate_fit='moderate_fit', score=70, matched_skills=['Python', 'ML models'], missing_skills=['AWS', 'production deployment'], experience_level='mid', shortlist=True, decision_reason='The candidate demonstrates proficiency in Python and has experience building ML models, which aligns with key job requirements. However, the resume lacks explicit mention of AWS expertise and production deployment experience, which are critical for the role. The 3‑year experience is sufficient for a mid‑level position but does not fully satisfy the seniority expectations for a production‑focused ML Engineer.')


In [ ]:
print(result.model_dump_json(indent=2))

{
  "candidate_fit": "moderate_fit",
  "score": 70,
  "matched_skills": [
    "Python",
    "ML models"
  ],
  "missing_skills": [
    "AWS",
    "production deployment"
  ],
  "experience_level": "mid",
  "shortlist": true,
  "decision_reason": "The candidate demonstrates proficiency in Python and has experience building ML models, which aligns with key job requirements. However, the resume lacks explicit mention of AWS expertise and production deployment experience, which are critical for the role. The 3‑year experience is sufficient for a mid‑level position but does not fully satisfy the seniority expectations for a production‑focused ML Engineer."
}
